First install the dependency

In [ ]:
pip install redshift-connector

Import all dependencies are needed

In [18]:
import csv
from zipfile import ZipFile, ZipInfo
from io import BytesIO, StringIO
from typing import IO, List
from datetime import datetime
from sys import exit, stdout
import logging
from redshift_connector import connect, Connection
import os

In [19]:
def redshift_open_connection(host: str, port: int, database: str, user: str, password: str) -> Connection:
    conn = connect(
        host=host,
        database=database,
        user=user,
        password=password,
        port=port
    )
    return conn

def redshift_open_connection_by_dict(dict_secret: dict, database: str = None) -> Connection:
    logger = logging.getLogger()
    logger.debug(f'Openning connection by secret dict')

    if database is None:
        database = dict_secret['dbname']


    return redshift_open_connection(
        dict_secret['hostname'],
        dict_secret['port'],
        database,
        dict_secret['user'],
        dict_secret['password']
    )

def redshift_get_rows_result_query_by_dict(dict_secret: dict, str_query: str, database: str = None) -> list:
    logger = logging.getLogger()
    conn = redshift_open_connection_by_dict(dict_secret, database)
    conn.autocommit = False
    cur = conn.cursor()
    
    logger.debug(f'Executing query')
    result = []
    cur.execute(str_query)
    cols = [a[0] for a in cur.description]
    for row in cur.fetchall():
        result.append({a: b for a,b in zip(cols, row)})
                
    cur.close()
    conn.close()
    
    return result
    
def list_dict_to_csv_bytes(lista: List[dict], delimiter: str = '|', quotechar: str = '"', quoting: int = csv.QUOTE_NONNUMERIC, lineterminator: str = '\r\n', list_keys: list = None, upper_header: bool = True, generate_header: bool = True) -> BytesIO:
    logger = logging.getLogger()
    line_header = None
    if generate_header:
        if (list_keys is None or isinstance(list_keys, list) == False or (isinstance(list_keys, list) and len(list_keys) == 0)) and lista is not None and len(lista) > 0:
            list_keys = list(lista[0].keys())

        if (isinstance(list_keys, list) and len(list_keys) > 0):
            if upper_header:
                list_keys = [k.upper() for k in list_keys]

            if quoting in [csv.QUOTE_NONNUMERIC, csv.QUOTE_ALL]:
                list_keys = [f'{quotechar}{k}{quotechar}' for k in list_keys]

            line_header = f"{delimiter.join(list_keys)}{lineterminator}"

    writer_file = StringIO()

    # Create the CSV DictWriter
    dict_writer = csv.DictWriter(writer_file, list(lista[0].keys()), delimiter=delimiter, quotechar=quotechar, quoting=quoting, lineterminator=lineterminator)

    if generate_header and line_header is not None and isinstance(line_header, str) and len(line_header) > 0:
        writer_file.write(line_header)

    # Write the rows
    dict_writer.writerows(lista)

    # Convert StringIO to BytesIO
    bytes_io = BytesIO()
    bytes_io.write(writer_file.getvalue().encode('utf-8'))
    bytes_io.seek(0)
    
    return bytes_io

def save_bytesio_csv(bytes_io: BytesIO, save_path: str):
    """
    Saves a BytesIO CSV object to a specified file path.

    Args:
        bytes_io (BytesIO): The BytesIO object containing CSV data.
        save_path (str): The file path where the CSV should be saved.
    """
    # Ensure the directory exists
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Save the BytesIO content to the specified path
    with open(save_path, 'wb') as f:
        f.write(bytes_io.getvalue())

Set your credencials and root path to store files

In [27]:
dict_secret = {
    "dbname": "datalake_dw",
    "port": 5439,
    "hostname": "asgard-redshift-production.cmqegk5gj3mi.sa-east-1.redshift.amazonaws.com",
    "user": "herculano_cunha",
    "password": ""
}
str_path_save = "/home/jovyan/work/files"

In [22]:
list_of_querys = [
    {'query': "select getdate() as current_date_query_1", 'filename': 'filename1.csv'},
      {'query': "select getdate() as current_date_query_2", 'filename': 'filename2.csv'}
]

In [29]:
for dict_query in list_of_querys:
    print(f"Selecting rows on redshift for query: {dict_query.get('query')}")
    list_dicts_rows = redshift_get_rows_result_query_by_dict(dict_secret, dict_query.get('query'))
    print("Row Getted")

    print(f"Generating csv UTF-8")
    file_bytes = list_dict_to_csv_bytes(lista=list_dicts_rows)
    print("File bytes UTF-8 generated")

    print(f"Saving File: {dict_query.get('filename')}")
    save_bytesio_csv(file_bytes, f"{str_path_save}/{dict_query.get('filename')}")
    print(f"{str_path_save}/{dict_query.get('filename')} - saved")
    print()
    print('------------------------')
    print()
    

Selecting rows on redshift for query: select getdate() as current_date_query_1
Row Getted
Generating csv UTF-8
File bytes UTF-8 generated
Saving File: filename1.csv
/home/jovyan/work/files/filename1.csv - saved

------------------------

Selecting rows on redshift for query: select getdate() as current_date_query_2
Row Getted
Generating csv UTF-8
File bytes UTF-8 generated
Saving File: filename2.csv
/home/jovyan/work/files/filename2.csv - saved

------------------------



In [26]:
print('All files saved')

'/home/jovyan/work'